# Ground Truth : démarrage sur Colab (GPU T4)

1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Exécution > **Tout exécuter**. Compter ~20 min la première fois (corpus, puis 4,7 Go de poids du générateur et 8 Go pour le juge).

Si Colab demande l'accès au secret HF_TOKEN, tu peux cliquer sur Annuler (ou ajouter ton jeton Hugging Face dans Secrets pour télécharger plus vite).

Le notebook télécharge le corpus, lance la baseline sur 20 questions de validation et les note avec le juge officiel.

In [ ]:
# >>> Remplace par l'adresse de TON dépôt GitHub (celui où tu as poussé le dossier src/) <<<
REPO_URL = "https://github.com/VOTRE_COMPTE/VOTRE_DEPOT.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

%cd /content
!test -d {REPO_DIR} || git clone -q {REPO_URL}
!git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q sentence-transformers rank_bm25

> Avant de lancer : fais `git add`, `git commit` et `git push` depuis VS Code, sinon Colab n'aura pas ta dernière version de `src/`. Si ton dépôt est privé, utilise une adresse avec jeton : `https://<TOKEN>@github.com/compte/depot.git`.

In [ ]:
!python corpus/download_corpus.py

In [ ]:
import subprocess
try:
    sortie = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                                      "--format=csv,noheader,nounits"], text=True).strip().splitlines()[0]
    nom, total, utilise = [x.strip() for x in sortie.split(",")]
    print(f"{nom} : {utilise} Mo utilisés sur {total} Mo")
    if int(utilise) > 1024:
        print("La mémoire GPU est déjà occupée : Exécution > Déconnecter et supprimer l'environnement "
              "d'exécution, puis Tout exécuter.")
except (FileNotFoundError, subprocess.CalledProcessError):
    print("Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4.")

## Le générateur officiel

Le test se lance dans un **processus séparé** (`!python -m groundtruth.generator`) : à la fin du processus, la VRAM est libérée. Ne garde pas le générateur chargé dans le noyau du notebook pendant qu'un script le recharge : le modèle occupe environ 7,3 Go, deux exemplaires ne tiennent pas sur un T4, et `del` + `torch.cuda.empty_cache()` ne les libèrent pas vraiment (d'où l'erreur « CUDA out of memory » à l'étape suivante).

Dans ton propre code, tu utiliseras la classe ainsi :

```python
from groundtruth.generator import OfficialGenerator

gen = OfficialGenerator(backend="hf")
gen.start_question("q1")
msgs = [{"role": "system", "content": "Réponds en une phrase, en français."},
        {"role": "user", "content": "Quelle est la capitale de la France ?"}]
print(gen.count_tokens(msgs), "tokens")
print(gen.chat(msgs, max_new_tokens=50))
```

In [ ]:
!python -m groundtruth.generator

## Pipeline hybride sur 20 questions de validation

### Vérifier le bloc 1 (les 4 fonctions à écrire) et les tests

In [ ]:
!python tests/test_block1.py
!python tests/test_bm25_vs_ref.py

In [ ]:
!python -m src.pipeline index --pdf-dir corpus/pdf --index-dir index/
import json
v = json.load(open('data/validation_v1.json'))
v['questions'] = v['questions'][:20]
json.dump(v, open('runs_val20.json', 'w'), ensure_ascii=False)
!python -m src.pipeline answer --index-dir index/ --questions runs_val20.json --out runs/val20.json --backend hf

In [ ]:
!python -m groundtruth.evaluate --pred runs/val20.json --gold runs_val20.json --judge hf

In [ ]:
!python diagnose.py --pred runs/val20.json --gold runs_val20.json

## Évaluer la recherche seule (100 questions, sans générateur) et lancer les ablations

`run_ablation.sh` compare BM25 seul, dense seul, hybride, avec et sans reranker, puis estime le seuil d'abstention.

In [ ]:
!python -m src.pipeline answer --index-dir index/ --questions data/validation_v1.json --out runs/ret.json --retrieval-only --k 20
!python eval_retrieval.py --pred runs/ret.json --gold data/validation_v1.json

In [ ]:
!bash run_ablation.sh index/ data/validation_v1.json

## Run complet avec seuil d'abstention

Mets dans `TAU` la valeur trouvée par `calibrate.py` (0.0 = pas de seuil).

In [ ]:
TAU = 0.0
!python -m src.pipeline answer --index-dir index/ --questions data/validation_v1.json --out runs/full.json --backend hf --tau {TAU}
!python -m groundtruth.evaluate --pred runs/full.json --gold data/validation_v1.json --judge hf
!python diagnose.py --pred runs/full.json --gold data/validation_v1.json

## Palier 3 : mise à jour du corpus

`update` ajoute les 20 documents v2 à l'index existant, sans tout réindexer.

In [ ]:
!python -m src.pipeline update --index-dir index/ --pdf-dir corpus/pdf_v2